# 06 - Export Model

This notebook exports the trained model artifacts into the project’s model directory so that the existing Healix runtime can load them.

In [1]:
import sys
from pathlib import Path

_nb = Path.cwd().resolve()
for _c in [_nb, _nb.parent, _nb.parent.parent]:
    if (_c / "app" / "main.py").exists():
        sys.path.insert(0, str(_c))
        break

from training.notebooks.bootstrap import require_ddxplus

ctx = require_ddxplus()
repo_root = ctx.repo_root
raw_path = ctx.raw_train_csv
processed_dir = ctx.processed_dir
training_models_dir = ctx.training_models_dir
runtime_models_dir = ctx.runtime_models_dir

import json

import joblib
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight

X_train = pd.read_csv(processed_dir / "X_train.csv")
y_train = pd.read_csv(processed_dir / "y_train.csv")["target"].astype(np.int16)
print("Export pipeline ready | train shape:", X_train.shape)


Export pipeline ready | train shape: (15000, 225)


In [ ]:
# Train the final XGBoost model (same configuration as 04_train_models.ipynb)
# and export it alongside the feature names / label encoder saved by notebook 03.
sample_weight = compute_sample_weight("balanced", y_train)

model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1,
)
model.fit(X_train, y_train, sample_weight=sample_weight)

runtime_model_path = runtime_models_dir / "xgboost_model.pkl"
joblib.dump(model, runtime_model_path)

print("Model trained and exported to:")
print(" ", runtime_model_path)

In [3]:
# Verify the exported model reloads and matches the training feature count
reloaded_model = joblib.load(runtime_model_path)
print("Reloaded model n_features_in_:", reloaded_model.n_features_in_)
print("Matches training feature count:", reloaded_model.n_features_in_ == X_train.shape[1])


Reloaded model n_features_in_: 225
Matches training feature count: True


## Export location

The exported artifacts — feature names / label encoder from notebook 03, and the XGBoost model trained above — are saved to the repository-level `models/` directory that `app/infrastructure/model_loader.py` loads at runtime.